In [1]:
import pandas as pd
import numpy as np

# ---------------------------
# Load Data
# ---------------------------
df = pd.read_csv('Nat_Gas.csv')

df['Dates'] = pd.to_datetime(df['Dates'])
df = df.sort_values('Dates')

df['Date_Num'] = df['Dates'].map(pd.Timestamp.toordinal)
df['Month'] = df['Dates'].dt.month

monthly_avg = df.groupby('Month')['Prices'].mean()

monthly_trend = (
    df['Prices'].iloc[-1]
    - df['Prices'].iloc[0]
) / (len(df)-1)

last_date = df['Dates'].max()

# ---------------------------
# Price Estimation Function
# ---------------------------

def get_historical_price(date):
    date = pd.to_datetime(date)
    date_num = date.toordinal()

    price = np.interp(
        date_num,
        df['Date_Num'],
        df['Prices']
    )

    return price


def get_future_price(date):
    date = pd.to_datetime(date)

    month = date.month

    base_price = monthly_avg[month]

    months_ahead = (
        (date.year - last_date.year) * 12
        + (date.month - last_date.month)
    )

    forecast = (
        base_price
        + months_ahead * monthly_trend
    )

    return forecast


def get_price(date):
    date = pd.to_datetime(date)

    if date <= last_date:
        return get_historical_price(date)
    else:
        return get_future_price(date)

# ---------------------------
# Contract Pricing Function
# ---------------------------

def price_contract(
        injection_date,
        withdrawal_date,
        quantity,
        storage_cost_per_month,
        injection_cost,
        withdrawal_cost,
        transport_cost):

    injection_date = pd.to_datetime(injection_date)
    withdrawal_date = pd.to_datetime(withdrawal_date)

    buy_price = get_price(injection_date)
    sell_price = get_price(withdrawal_date)

    storage_months = (
        (withdrawal_date.year - injection_date.year) * 12
        + (withdrawal_date.month - injection_date.month)
    )

    revenue = (
        sell_price - buy_price
    ) * quantity

    total_cost = (
        storage_months * storage_cost_per_month
        + injection_cost
        + withdrawal_cost
        + 2 * transport_cost
    )

    contract_value = revenue - total_cost

    return round(contract_value, 2)

C:\Users\hp\AppData\Local\Temp\ipykernel_5440\4107046715.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Dates'] = pd.to_datetime(df['Dates'])


In [3]:
value = price_contract(
    injection_date='2024-05-31',
    withdrawal_date='2024-12-31',
    quantity=1000000,
    storage_cost_per_month=100000,
    injection_cost=10000,
    withdrawal_cost=10000,
    transport_cost=50000
)

print(f'Contract Value: ${value:,.2f}')

Contract Value: $-411,489.36


I used historical natural gas prices to estimate prices for any given date. For dates within the dataset, linear interpolation was applied. For future dates, I combined seasonal monthly averages with the overall price trend to forecast prices. The storage contract value was calculated as the profit from buying and selling gas minus all associated costs, including storage, injection, withdrawal, and transportation costs.
